# Dane dla analityka: transakcje przypadków i dwa foldery w GCS

Jednorazowo. Analityk oznacza **przypadki** (grupy podejrzanych transakcji), więc oprócz kolejki potrzebuje surowych transakcji każdego przypadku. Zamrożony zbiór `samld-case` ma tylko cechy przypadków, więc ten notebook odtwarza przypisanie transakcja → przypadek tą samą funkcją łączenia co freeze i sprawdza, że numery przypadków się zgadzają.

Wynik w GCS (poza zamrożonym zbiorem, nie wpływa na SHA ani na trening):
- `analyst/reference/samld-case/case_transactions.parquet`: transakcje przypadków, bez etykiet (`Is_laundering`, `Laundering_type` są usunięte);
- `analyst/do_oznaczenia/`: tu pipeline odkłada partie do oznaczenia (`<przebieg>/fold<k>/kolejka.csv`, `transakcje.csv`, `README.txt`);
- `analyst/oznaczone/`: tu analityk odkłada wypełniony `kolejka.csv` pod tą samą ścieżką partii.

Kernel: `fsl (py3.10)`. Czas: wczytanie CSV kilka minut, jak przy freeze.

In [ ]:
# KOMORKA 1 — config i funkcje laczenia z bootstrapu (te same, ktorymi zrobiono freeze)
import importlib.util, json, sys
from pathlib import Path

HOME = Path.home()
REPO = HOME / "mlops-meta-learning/fsl-projekt"
CFG = json.loads((REPO / "configs/pipeline_config_samld.json").read_text())
PROJECT, BUCKET, DS = CFG["project"], CFG["bucket"], "samld-case"
CSV = HOME / "data_cache/saml_d_source/SAML-D.csv"
assert CSV.exists(), f"brak {CSV} (ten sam plik co przy freeze)"

spec = importlib.util.spec_from_file_location("boot", REPO / "scripts/bootstrap_fsl_data.py")
boot = importlib.util.module_from_spec(spec); sys.modules["boot"] = boot; spec.loader.exec_module(boot)
print("projekt:", PROJECT, "| bucket:", BUCKET, "| okno laczenia:", boot.SAMLD_CASE_LINK_DAYS, "dni")

In [ ]:
# KOMORKA 2 — podejrzane transakcje i ich przypadki (ta sama kolejnosc wierszy co we freeze)
import pandas as pd

S, n_all = boot._samld_case_load_flagged(CSV)           # dokladnie jak freeze
S["case"] = boot._samld_case_link(S, boot.SAMLD_CASE_LINK_DAYS)
# godzina transakcji: ta sama kolejnosc czytania i ten sam filtr -> wiersze 1:1 z S
times = [ch.loc[ch["Is_laundering"] == 1, "Time"]
         for ch in pd.read_csv(CSV, usecols=["Time", "Is_laundering"], chunksize=2_000_000)]
S["Time"] = pd.concat(times, ignore_index=True).astype(str).to_numpy()
print(f"transakcji: {n_all:,} | podejrzanych: {len(S):,} | przypadkow: {S['case'].nunique():,}")

In [ ]:
# KOMORKA 3 — weryfikacja: numery przypadkow musza sie zgadzac z zamrozonym zbiorem (loader pipeline'u)
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders

cfg = TrainConfig(project=PROJECT, bucket=BUCKET, dataset=DS, seed=0, fold=0,
                  n_way=4, k_shot=5, query=5, log_to_vertex=False)
load_frozen, _ = get_loaders(cfg)
bundle, sha = load_frozen(cfg)
C = bundle["windows"]                                   # jeden wiersz na przypadek
g = S.groupby("case")
check = pd.DataFrame({
    "n_tx": g.size(),
    "n_acc": g.apply(lambda d: len(set(d["Sender_account"]) | set(d["Receiver_account"]))),
})
merged = C[["case", "n_tx", "n_acc"]].merge(check, left_on="case", right_index=True,
                                             how="left", suffixes=("_freeze", "_tu"))
bad = merged[(merged["n_tx_freeze"] != merged["n_tx_tu"]) | (merged["n_acc_freeze"] != merged["n_acc_tu"])]
print(f"sha zbioru {sha[:12]} | przypadkow w zbiorze: {len(C):,} | niezgodnych: {len(bad)}")
assert bad.empty, "numery przypadkow sie nie zgadzaja z freeze - STOP, wklej mi ten output"

In [ ]:
# KOMORKA 4 — zapis do GCS: transakcje przypadkow (bez etykiet) + README w obu folderach
from google.cloud import storage

keep = set(C["case"].astype(int))
T = S[S["case"].isin(keep)]
T = pd.DataFrame({
    "przypadek": T["case"].astype(int),
    "data": pd.to_datetime(T["Date"]).dt.strftime("%Y-%m-%d"),
    "godzina": T["Time"],
    "nadawca": T["Sender_account"].astype("int64"),
    "odbiorca": T["Receiver_account"].astype("int64"),
    "kwota": T["Amount"].round(2),
    "waluta_platnosci": T["Payment_currency"],
    "waluta_odbioru": T["Received_currency"],
    "typ_platnosci": T["Payment_type"],
    "lokalizacja_nadawcy": T["Sender_bank_location"],
    "lokalizacja_odbiorcy": T["Receiver_bank_location"],
}).sort_values(["przypadek", "data", "godzina"]).reset_index(drop=True)
assert not {"Is_laundering", "Laundering_type"} & set(T.columns)

local = HOME / "data_cache/case_transactions.parquet"
T.to_parquet(local, index=False)
bucket = storage.Client(project=PROJECT).bucket(BUCKET)
bucket.blob(f"analyst/reference/{DS}/case_transactions.parquet").upload_from_filename(str(local))

bucket.blob("analyst/do_oznaczenia/README.txt").upload_from_string(
    "Partie przypadkow do oznaczenia, odkladane przez pipeline.\n"
    "Sciezka partii: do_oznaczenia/<przebieg>/fold<k>/ (kolejka.csv, transakcje.csv, README.txt).\n"
    "Wypelniony kolejka.csv odklada sie do oznaczone/<przebieg>/fold<k>/kolejka.csv.\n",
    content_type="text/plain; charset=utf-8")
bucket.blob("analyst/oznaczone/README.txt").upload_from_string(
    "Wypelnione kolejki od analityka. Ta sama sciezka partii co w do_oznaczenia/:\n"
    "oznaczone/<przebieg>/fold<k>/kolejka.csv, z uzupelniona kolumna typologia_analityka.\n",
    content_type="text/plain; charset=utf-8")
print(f"zapisane: {len(T):,} transakcji z {T['przypadek'].nunique():,} przypadkow")
print(f"gs://{BUCKET}/analyst/reference/{DS}/case_transactions.parquet")
print(f"foldery: gs://{BUCKET}/analyst/do_oznaczenia/  i  gs://{BUCKET}/analyst/oznaczone/")